# Spark Cleaning
Read the 4 raw TSVs with Spark, clean them, combine into one table, and save as Parquet.

## 1. Start Spark

In [1]:
import os
import warnings
from functools import reduce  

warnings.filterwarnings("ignore")
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYTHONWARNINGS"] = "ignore"

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType 

spark = (
    SparkSession.builder
    .appName("amazon-reviews")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "32")
    .config("spark.sql.ansi.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print(f"Spark {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/24 22:28:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.2.1


## 2. Find the raw files (reuses the Stage 1 download, no re-download)

In [2]:
import os
os.environ["KAGGLEHUB_CACHE"] = "./data"
import kagglehub

DATASET = "cynthiarempel/amazon-us-customer-reviews-dataset"
FILES = {
    "Books":         "amazon_reviews_us_Books_v1_02.tsv",
    "Music":         "amazon_reviews_us_Music_v1_00.tsv",
    "Video_DVD":     "amazon_reviews_us_Video_DVD_v1_00.tsv",
    "Digital_Ebook": "amazon_reviews_us_Digital_Ebook_Purchase_v1_01.tsv",
}

paths = {cat: kagglehub.dataset_download(DATASET, path=name) for cat, name in FILES.items()}
for cat, p in paths.items():
    print(cat, "->", p)

Books -> ./data/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/versions/9/amazon_reviews_us_Books_v1_02.tsv
Music -> ./data/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/versions/9/amazon_reviews_us_Music_v1_00.tsv
Video_DVD -> ./data/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/versions/9/amazon_reviews_us_Video_DVD_v1_00.tsv
Digital_Ebook -> ./data/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/versions/9/amazon_reviews_us_Digital_Ebook_Purchase_v1_01.tsv


## 3. Read with an explicit schema
Bad rows (for example a stray tab that shifts the columns) turn into `null` instead of crashing, and we drop them in the next step.

In [3]:
SCHEMA = StructType([
    StructField("marketplace",       StringType(),  True),
    StructField("customer_id",       LongType(),    True),
    StructField("review_id",         StringType(),  True),
    StructField("product_id",        StringType(),  True),
    StructField("product_parent",    LongType(),    True),
    StructField("product_title",     StringType(),  True),
    StructField("product_category",  StringType(),  True),
    StructField("star_rating",       IntegerType(), True),
    StructField("helpful_votes",     IntegerType(), True),
    StructField("total_votes",       IntegerType(), True),
    StructField("vine",              StringType(),  True),
    StructField("verified_purchase", StringType(),  True),
    StructField("review_headline",   StringType(),  True),
    StructField("review_body",       StringType(),  True),
    StructField("review_date",       StringType(),  True),
])

def read_category(path):
    return (spark.read
            .option("sep", "\t")
            .option("header", True)
            .option("quote", "")            # the TSVs are not quoted
            .option("mode", "PERMISSIVE")
            .schema(SCHEMA)
            .csv(path))

## 4. Cleaning function
Steps: drop malformed rows, drop duplicate `review_id`s, strip HTML from the text, merge headline and body, drop very short reviews, parse the date, and keep only the columns we need.

In [4]:
def clean(df, category):
    df = df.withColumn("category", F.lit(category))

    # 1. drop rows that failed to parse or have missing key fields
    df = df.filter(
        F.col("review_id").isNotNull()
        & F.col("star_rating").between(1, 5)
        & F.col("review_body").isNotNull()
        & F.col("review_date").isNotNull()
    )

    # 2. drop duplicate reviews
    df = df.dropDuplicates(["review_id"])

    # 3. build one clean text field: headline + body
    text = F.concat_ws(" ", F.coalesce(F.col("review_headline"), F.lit("")), F.col("review_body"))
    text = F.regexp_replace(text, r"<[^>]+>", " ")     # HTML tags such as <br />
    text = F.regexp_replace(text, r"&#?\w+;", " ")     # HTML entities such as &#34;
    text = F.regexp_replace(text, r"\s+", " ")         # collapse whitespace
    df = df.withColumn("review_text", F.lower(F.trim(text)))

    # 4. drop very short reviews (fewer than 3 words)
    df = df.withColumn("word_count", F.size(F.split("review_text", " ")))
    df = df.filter(F.col("word_count") >= 3)

    # 5. dates and flags
    df = df.withColumn("review_date", F.to_date("review_date", "yyyy-MM-dd"))
    df = df.filter(F.col("review_date").isNotNull())
    df = df.withColumn("review_year", F.year("review_date"))
    df = df.withColumn("verified_purchase", F.col("verified_purchase") == "Y")

    return df.select(
        "review_id", "category", "product_id", "customer_id",
        "star_rating", "helpful_votes", "total_votes", "verified_purchase",
        "review_date", "review_year", "review_text", "word_count",
    )

## 5. Run on all four files and save as Parquet
This is the slow step (several minutes). Parquet is partitioned by `category`.

In [5]:
OUT = "data/processed/reviews.parquet"

cleaned = [clean(read_category(paths[cat]), cat) for cat in FILES]
all_reviews = reduce(lambda a, b: a.unionByName(b), cleaned)

(all_reviews.write
    .mode("overwrite")
    .partitionBy("category")
    .parquet(OUT))
print("Saved", OUT)

Saved data/processed/reviews.parquet


## 6. Check the result (rows kept vs. the Stage 1 line counts)

In [8]:
df = spark.read.parquet(OUT)

stage1_rows = {"Books": 3105520, "Music": 4751577, "Video_DVD": 5069140, "Digital_Ebook": 5101693}
kept = {r["category"]: r["count"] for r in df.groupBy("category").count().collect()}

print(f"{'Category':<15}{'Stage 1 rows':>15}{'Clean rows':>15}{'Kept':>8}")
for cat, raw in stage1_rows.items():
    print(f"{cat:<15}{raw:>15,}{kept[cat]:>15,}{kept[cat]/raw:>8.1%}")
total_clean = sum(kept.values())
print(f"{'TOTAL':<15}{sum(stage1_rows.values()):>15,}{total_clean:>15,}{total_clean/sum(stage1_rows.values()):>8.1%}")

Category          Stage 1 rows     Clean rows    Kept
Books                3,105,520      3,105,433  100.0%
Music                4,751,577      4,748,339   99.9%
Video_DVD            5,069,140      5,063,668   99.9%
Digital_Ebook        5,101,693      5,101,681  100.0%
TOTAL               18,027,930     18,019,121  100.0%


In [9]:
print("\nRating distribution:")
df.groupBy("star_rating").count().orderBy("star_rating").show()


Rating distribution:
+-----------+--------+
|star_rating|   count|
+-----------+--------+
|          1| 1028038|
|          2|  760850|
|          3| 1428976|
|          4| 3370648|
|          5|11430609|
+-----------+--------+



In [10]:
print("Reviews per year:")
df.groupBy("review_year").count().orderBy("review_year").show(30)

Reviews per year:
+-----------+-------+
|review_year|  count|
+-----------+-------+
|       1995|    199|
|       1996|   4677|
|       1997|  36722|
|       1998| 157484|
|       1999| 380500|
|       2000| 854878|
|       2001| 738302|
|       2002| 747792|
|       2003| 830643|
|       2004| 858296|
|       2005| 857668|
|       2006| 396505|
|       2007| 475039|
|       2008| 445926|
|       2009| 487012|
|       2010| 535116|
|       2011| 807549|
|       2012|2111741|
|       2013|4377412|
|       2014|1604621|
|       2015|1311039|
+-----------+-------+



In [11]:
df.printSchema()

root
 |-- review_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- star_rating: integer (nullable = true)
 |-- helpful_votes: integer (nullable = true)
 |-- total_votes: integer (nullable = true)
 |-- verified_purchase: boolean (nullable = true)
 |-- review_date: date (nullable = true)
 |-- review_year: integer (nullable = true)
 |-- review_text: string (nullable = true)
 |-- word_count: integer (nullable = true)
 |-- category: string (nullable = true)



In [12]:
df.show(3, truncate=80)

+--------------+----------+-----------+-----------+-------------+-----------+-----------------+-----------+-----------+--------------------------------------------------------------------------------+----------+--------+
|     review_id|product_id|customer_id|star_rating|helpful_votes|total_votes|verified_purchase|review_date|review_year|                                                                     review_text|word_count|category|
+--------------+----------+-----------+-----------+-------------+-----------+-----------------+-----------+-----------+--------------------------------------------------------------------------------+----------+--------+
|R1002GW8P5Q4OY|B00005A1PR|   43369200|          4|            1|          1|            false| 2002-05-02|       2002|good girls can win (or at least get nominated)! i liked almost every song off...|        89|   Music|
|R1002P9DWHQSBB|B00000AYPS|   25778231|          5|            0|          0|            false| 2005-12-02|       20

26/11/25 22:48:58 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:131)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(Blo

## 7. Save a 10% sample
Training on all ~18M reviews is slow on a laptop. This sample (about 1.8M rows) is what we will use for modeling in Stage 3. The full data stays available for EDA.

In [7]:
SAMPLE_OUT = "data/processed/reviews_sample.parquet"

sample = df.sample(fraction=0.10, seed=42)
sample.write.mode("overwrite").partitionBy("category").parquet(SAMPLE_OUT)

print("Saved", SAMPLE_OUT, "| rows:", spark.read.parquet(SAMPLE_OUT).count())

Saved data/processed/reviews_sample.parquet | rows: 1802532


**Done with Stage 2.** Stage 3 reads `data/processed/reviews_sample.parquet` for EDA and modeling.